# TPAMI 2025 RDHP/RDP experiment notebook

This notebook is prepared for Google Colab. It clones the private `my-thesis` repository with its submodules, runs the RDHP reconstruction self-test and synthetic smoke test, and provides an optional real CIFAR-10 ResNet-32 pilot comparing RDP and RDHP from one shared dense checkpoint.

The real experiment is disabled by default. Set `RUN_REAL_DATA = True` only after reviewing the configuration and ensuring that the Colab runtime has enough time and GPU memory.

## Before running

1. In Colab, add a GitHub personal-access token with private-repository read access as a Colab Secret named `GITHUB_TOKEN`, or enter it when prompted.
2. Use a GPU runtime for the real experiment (`Runtime` -> `Change runtime type` -> `T4 GPU` or better).
3. The notebook never stores the token in the notebook and resets the Git remote to the public URL after cloning.

This is a reconstruction of the paper's unstructured RDHP/RDP path on ResNet-32/CIFAR-10. It is not a full reproduction of the structured, ViT, 3D-detector, or hardware-aware experiments.

In [5]:
from pathlib import Path
import getpass
import json
import os
import shutil
import subprocess
import sys

OWNER = 'hamidrg20001379'
REPO_NAME = 'my-thesis'
REPO_URL = f'https://github.com/{OWNER}/{REPO_NAME}.git'
REPO = Path('/content') / REPO_NAME

if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'{REPO} exists but is not a Git checkout. Remove it and rerun this cell.')

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not token:
    token = getpass.getpass('GitHub token with private-repository read access: ')
if not token:
    raise RuntimeError('A GitHub token is required to clone the private repository.')

# Keep the token out of process arguments and tracebacks. Git requests it
# through this short-lived askpass helper instead of an authenticated URL.
askpass = Path('/tmp/github-askpass.sh')
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    "  *Username*) printf '%s\n' x-access-token ;;\n"
    "  *Password*) printf '%s\n' \"$GITHUB_TOKEN_FOR_ASKPASS\" ;;\n"
    "  *) printf '\n' ;;\n"
    'esac\n'
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
    'GITHUB_TOKEN_FOR_ASKPASS': token,
})
try:
    if not REPO.exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', REPO_URL, str(REPO)], env=git_env, check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'], env=git_env, check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(f'GitHub clone failed with exit code {exc.returncode}. Verify the token has access to {OWNER}/{REPO_NAME} and its private submodules.') from None
finally:
    askpass.unlink(missing_ok=True)

print('Repository:', REPO)
print(subprocess.run(['git', '-C', str(REPO), 'status', '--short', '--branch'], check=True, capture_output=True, text=True).stdout)

Repository: /content/my-thesis
## master...origin/master



In [6]:
# Colab normally provides compatible torch/torchvision builds. Avoid upgrading them here.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas'], check=True)
import numpy as np
import pandas as pd
import torch
import torchvision

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Python: 3.13.16
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
CUDA available: True
GPU: Tesla T4


In [7]:
EXPERIMENTS = REPO / 'working' / 'experiments'
RDP_SCRIPT = EXPERIMENTS / 'rdp_resnet32_adapter.py'
RDHP_SCRIPT = EXPERIMENTS / 'rdhp_resnet32_adapter.py'
sys.path.insert(0, str(EXPERIMENTS))

# Main configuration. The real run is intentionally opt-in.
RUN_REAL_DATA = False
SEED = 42
TRAIN_SAMPLES = 5000
CALIB_SAMPLES = 1024
TEST_SAMPLES = 1000
BATCH_SIZE = 128
DENSE_EPOCHS = 20
FINETUNE_EPOCHS = 5
TARGET_SPARSITY = 0.50
CURVE_POINTS = 100
FISHER_PROBES = 2

RUN_ROOT = Path('/content') / f'rdhp_rdp_runs_seed_{SEED}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Run directory:', RUN_ROOT)

Run directory: /content/rdhp_rdp_runs_seed_42


## 1. Software checks

The RDHP self-test checks empirical-Fisher estimation and verifies that the overlapping update agrees with direct quadratic evaluation on a tiny model.

In [8]:
subprocess.run([sys.executable, str(RDHP_SCRIPT), '--self-test'], check=True)
subprocess.run([sys.executable, str(RDP_SCRIPT), '--self-test'], check=True)

CompletedProcess(args=['/usr/bin/python3', '/content/my-thesis/working/experiments/rdp_resnet32_adapter.py', '--self-test'], returncode=0)

## 2. End-to-end synthetic smoke test

This validates execution and output schemas only. Synthetic accuracy is not research evidence.

In [9]:
SMOKE_ROOT = RUN_ROOT / 'synthetic_smoke'
common_smoke = [
    '--synthetic', '--train-samples', '32', '--calib-samples', '16',
    '--test-samples', '16', '--batch-size', '8', '--train-epochs', '1',
    '--finetune-epochs', '1', '--target-sparsity', '0.50',
    '--curve-points', '3', '--budget-bins', '1000', '--seed', str(SEED),
]
subprocess.run([sys.executable, str(RDP_SCRIPT), *common_smoke,
                '--worst-case-curve', '--monotonic-repair',
                '--output', str(SMOKE_ROOT / 'rdp')], check=True)
subprocess.run([sys.executable, str(RDHP_SCRIPT), *common_smoke,
                '--fisher-probes', '2',
                '--output', str(SMOKE_ROOT / 'rdhp')], check=True)

smoke_rows = []
for method in ('rdp', 'rdhp'):
    result_path = SMOKE_ROOT / method / 'result.json'
    result = json.loads(result_path.read_text())
    smoke_rows.append({
        'method': result['method'],
        'dense_accuracy': result['dense_accuracy'],
        'immediate_pruned_accuracy': result['immediate_pruned_accuracy'],
        'finetuned_pruned_accuracy': result['finetuned_pruned_accuracy'],
        'achieved_sparsity': result['allocation']['achieved_sparsity'],
    })
pd.DataFrame(smoke_rows)

,method,dense_accuracy,immediate_pruned_accuracy,finetuned_pruned_accuracy,achieved_sparsity
0,RDP_reconstruction,18.75,18.75,25.0,0.495907
1,RDHP_reconstruction,18.75,18.75,25.0,0.494436


## 3. Optional real CIFAR-10 pilot

Set `RUN_REAL_DATA = True` in the configuration cell before running the next cells. This cell downloads CIFAR-10, trains one common dense ResNet-32 checkpoint, and saves it for both methods. The pilot uses a 5,000-example training subset and 1,000 test examples by default; adjust the configuration for your approved protocol.

In [10]:
if RUN_REAL_DATA:
    from argparse import Namespace
    from rdp_resnet32_adapter import load_resnet32, make_loaders, train_epochs

    DATA_ROOT = Path('/content/data.cifar10')
    checkpoint_path = RUN_ROOT / 'dense_resnet32.pt'
    loader_args = Namespace(
        synthetic=False, data_root=str(DATA_ROOT), download=True,
        train_samples=TRAIN_SAMPLES, calib_samples=CALIB_SAMPLES,
        test_samples=TEST_SAMPLES, batch_size=BATCH_SIZE, seed=SEED,
    )
    loaders, split_info = make_loaders(loader_args)
    train_loader, calibration_loader, test_loader = loaders
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dense_model = load_resnet32().to(device)
    train_epochs(dense_model, train_loader, device, DENSE_EPOCHS, 0.1, SEED)
    torch.save({'model_state_dict': dense_model.state_dict(), 'split': split_info,
               'seed': SEED, 'device': str(device)}, checkpoint_path)
    print('Saved shared dense checkpoint:', checkpoint_path)
    print('Split:', split_info)
else:
    print('Real-data pilot disabled. Set RUN_REAL_DATA = True and rerun this cell.')

Real-data pilot disabled. Set RUN_REAL_DATA = True and rerun this cell.


In [11]:
if RUN_REAL_DATA:
    real_common = [
        '--data-root', str(DATA_ROOT), '--train-samples', str(TRAIN_SAMPLES),
        '--calib-samples', str(CALIB_SAMPLES), '--test-samples', str(TEST_SAMPLES),
        '--batch-size', str(BATCH_SIZE), '--target-sparsity', str(TARGET_SPARSITY),
        '--curve-points', str(CURVE_POINTS), '--finetune-epochs', str(FINETUNE_EPOCHS),
        '--dense-checkpoint', str(checkpoint_path), '--seed', str(SEED),
    ]
    rdp_output = RUN_ROOT / 'real_rdp'
    rdhp_output = RUN_ROOT / 'real_rdhp'
    subprocess.run([sys.executable, str(RDP_SCRIPT), *real_common,
                    '--worst-case-curve', '--monotonic-repair',
                    '--output', str(rdp_output)], check=True)
    subprocess.run([sys.executable, str(RDHP_SCRIPT), *real_common,
                    '--fisher-probes', str(FISHER_PROBES),
                    '--output', str(rdhp_output)], check=True)
    print('RDP and RDHP completed.')
else:
    print('Real-data pilot disabled.')

Real-data pilot disabled.


In [12]:
def load_result(path):
    return json.loads((Path(path) / 'result.json').read_text())

result_dirs = [SMOKE_ROOT / 'rdp', SMOKE_ROOT / 'rdhp']
if RUN_REAL_DATA:
    result_dirs += [RUN_ROOT / 'real_rdp', RUN_ROOT / 'real_rdhp']

rows = []
for directory in result_dirs:
    if not (directory / 'result.json').exists():
        continue
    result = load_result(directory)
    allocation = result.get('allocation', {})
    rows.append({
        'run': directory.name,
        'method': result.get('method'),
        'device': result.get('device'),
        'dense_accuracy': result.get('dense_accuracy'),
        'immediate_pruned_accuracy': result.get('immediate_pruned_accuracy'),
        'finetuned_pruned_accuracy': result.get('finetuned_pruned_accuracy'),
        'achieved_sparsity': allocation.get('achieved_sparsity'),
        'sparsity_error': allocation.get('sparsity_error'),
        'curve_or_allocation_seconds': result.get('elapsed_curve_and_allocation_seconds'),
    })

summary = pd.DataFrame(rows)
display(summary)
print('Artifacts are under:', RUN_ROOT)

,run,method,device,dense_accuracy,immediate_pruned_accuracy,finetuned_pruned_accuracy,achieved_sparsity,sparsity_error,curve_or_allocation_seconds
0,rdp,RDP_reconstruction,cuda,18.75,18.75,25.0,0.495907,-0.004093,3.830623
1,rdhp,RDHP_reconstruction,cuda,18.75,18.75,25.0,0.494436,-0.005564,0.507414


Artifacts are under: /content/rdhp_rdp_runs_seed_42


## Interpretation limits

- The RDHP implementation is a reconstruction using output-Jacobian empirical-Fisher probes; it is not verified author code.
- The default Fisher representation is diagonal for practical ResNet-32 memory use.
- The comparison must use achieved retained weights/sparsity, not only the requested target.
- Synthetic runs validate software execution only.
- A single pilot is not a paper reproduction. After the pilot, repeat with paired seeds and record checkpoint, data split, runtime, curve settings, and deviations.

Optional: copy `RUN_ROOT` to Google Drive after mounting it, so the artifacts survive Colab session shutdown.

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
from pathlib import Path
import shutil

src = Path("/content/rdhp_rdp_runs_seed_42")
dst = Path("/content/drive/MyDrive/rdhp_rdp_runs_seed_42")

if not src.exists():
    raise FileNotFoundError(f"Artifacts not found: {src}")

shutil.copytree(src, dst, dirs_exist_ok=True)
print("Saved to:", dst)

Saved to: /content/drive/MyDrive/rdhp_rdp_runs_seed_42
